# Customer Support Tone - Neural Network Regression

A frozen BERT encoder creates text features and a configurable MLP predicts labels from 1 to 5. The supplied `validation.csv` is the fixed development set used for MAE.

In [1]:
from pathlib import Path
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel, AutoTokenizer

SEED = 42
# Hyperparameters: change these values and rerun from this cell onward.
BATCH_SIZE = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
HIDDEN_DIMS = (256, 64)
DROPOUT = 0.25
MAX_EPOCHS = 100
PATIENCE = 15

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'train.csv').exists():
    DATA_DIR = DATA_DIR / 'customersupporttone'
train_df = pd.read_csv(DATA_DIR / 'train.csv')
dev_df = pd.read_csv(DATA_DIR / 'validation.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

def make_text(frame):
    return ('[CONVERSATION] ' + frame['conversation'].fillna('').astype(str) +
            ' [LAST_CUSTOMER] ' + frame['last_customer_message'].fillna('').astype(str)).tolist()

train_texts = make_text(train_df)
dev_texts = make_text(dev_df)
test_texts = make_text(test_df)
print(f'Train rows: {len(train_df)} | validation rows: {len(dev_df)} | test rows: {len(test_df)}')

c:\Users\bradm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train rows: 302 | validation rows: 103 | test rows: 101


## Frozen BERT embeddings

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert = AutoModel.from_pretrained('bert-base-uncased').to(device).eval()

def embed_texts(texts, batch_size=16):
    embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
        with torch.no_grad():
            embeddings.append(bert(**batch).last_hidden_state[:, 0, :].cpu())
    return torch.cat(embeddings).float()

X_train = embed_texts(train_texts)
X_dev = embed_texts(dev_texts)
X_test = embed_texts(test_texts)
print(f'Embedding shape: {tuple(X_train.shape)} | device: {device}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1507.08it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (302, 768) | device: cuda


## Train on train.csv and evaluate on validation.csv

In [3]:
class ToneRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dims[0], hidden_dims[1]), nn.ReLU(), nn.Dropout(dropout / 2),
            nn.Linear(hidden_dims[1], 1))

    def forward(self, features):
        return self.network(features)

train_dataset = TensorDataset(X_train, torch.tensor(train_df['label'].to_numpy(dtype='float32')).unsqueeze(1))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))
model = ToneRegressor(X_train.shape[1], HIDDEN_DIMS, DROPOUT).to(device)
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
best_weights, best_dev_mae, no_improvement = None, float('inf'), 0

def evaluate_dev():
    model.eval()
    with torch.no_grad():
        predictions = model(X_dev.to(device)).squeeze(1).cpu().numpy()
    return mean_absolute_error(dev_df['label'], np.clip(predictions, 1, 5))

for epoch in range(MAX_EPOCHS):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        criterion(model(batch_x), batch_y).backward()
        optimizer.step()
    dev_mae = evaluate_dev()
    if dev_mae < best_dev_mae:
        best_dev_mae, best_weights, no_improvement = dev_mae, copy.deepcopy(model.state_dict()), 0
    else:
        no_improvement += 1
    if (epoch + 1) % 10 == 0 or no_improvement == 0:
        print(f'Epoch {epoch + 1:03d} | validation MAE: {dev_mae:.4f}')
    if no_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch + 1}')
        break

model.load_state_dict(best_weights)
torch.save(model.state_dict(), DATA_DIR / 'best_customer_tone_regressor.pt')
print(f'Best validation MAE: {best_dev_mae:.4f}')

Epoch 001 | validation MAE: 1.7830
Epoch 002 | validation MAE: 0.9657
Epoch 003 | validation MAE: 0.9619
Epoch 004 | validation MAE: 0.9617
Epoch 005 | validation MAE: 0.9385
Epoch 007 | validation MAE: 0.9328
Epoch 009 | validation MAE: 0.9274
Epoch 010 | validation MAE: 0.9356
Epoch 012 | validation MAE: 0.9271
Epoch 013 | validation MAE: 0.9033
Epoch 018 | validation MAE: 0.9001
Epoch 019 | validation MAE: 0.8928
Epoch 020 | validation MAE: 0.9341
Epoch 022 | validation MAE: 0.8892
Epoch 024 | validation MAE: 0.8865
Epoch 030 | validation MAE: 0.9166
Early stopping at epoch 39
Best validation MAE: 0.8865


In [4]:
from sklearn.metrics import f1_score

model.eval()
with torch.no_grad():
    best_dev_predictions = model(X_dev.to(device)).squeeze(1).cpu().numpy()
dev_class_predictions = np.rint(np.clip(best_dev_predictions, 1, 5)).astype(int)
dev_macro_f1 = f1_score(dev_df['label'], dev_class_predictions, average='macro')
print(f'Best development macro-F1: {dev_macro_f1:.4f}')

Best development macro-F1: 0.2058


## Test submission

In [5]:
model.eval()
with torch.no_grad():
    test_predictions = model(X_test.to(device)).squeeze(1).cpu().numpy()
submission = pd.DataFrame({'id': test_df['id'], 'label': np.clip(test_predictions, 1, 5)})
submission_path = DATA_DIR / 'submission_nn.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} predictions to {submission_path}')
submission.head()

Saved 101 predictions to c:\Users\bradm\Documents\GitHub\ai-masters-workspace\ARI-510-Machine-Learning\Phase 2b\customersupporttone\submission_nn.csv


,id,label
0,18972,3.077774
1,49147,3.845341
2,86244,4.242389
3,98577,3.562607
4,99913,2.958538


In [6]:
if 'label' in test_df.columns:
    test_macro_f1 = f1_score(test_df['label'], np.rint(np.clip(test_predictions, 1, 5)).astype(int), average='macro')
    print(f'Test macro-F1: {test_macro_f1:.4f}')
else:
    print('Test macro-F1 unavailable: test.csv has no ground-truth label column.')

Test macro-F1 unavailable: test.csv has no ground-truth label column.
